# B2 · Points, lines and polygons: geometry with shapely

> **Time:** about 90 minutes  
> **You need:** Lesson B1 and all of Part A. You should know that x is longitude and y is latitude.  
> **By the end you can:**
> 1. build points, lines and polygons with **shapely**, always as (x, y) = (longitude, latitude),
> 2. read their coordinates, bounds and centroid, measure length, area and distance, and explain why these numbers are in degrees,
> 3. test how two shapes relate (contains, within, intersects), write shapes as WKT text, and draw them with GeoPandas.

In Lesson B1, a peak was two numbers in a table. In this lesson it becomes a **geometry**: an object that knows it is a point, and can measure itself and compare itself with other shapes. You will build the three shapes that all vector data is made of. Everest's summit becomes a point, the Everest trek a line, and a box around the Kathmandu Valley a polygon.

**Counting lines:** `fig, ax = plt.subplots()` and `plt.show()` do not count toward the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 12 of 21 · [Course home](../README.md)  
← Previous: [B1 · Maps, coordinates, latitude and longitude: placing Nepal on the Earth](B01_Maps_and_coordinates.ipynb) · Next: [B3 · GeoDataFrames and file formats: Nepal's districts as a table](B03_GeoDataFrames_and_file_formats.ipynb) →

## 1. Points

**shapely** is the Python library for geometry: points, lines and polygons on a flat x–y plane. It knows nothing about the Earth. To shapely, coordinates are plain numbers.

`Point(x, y)` makes a point. As in Lesson B1, **x is longitude and y is latitude**: `Point(86.925, 27.9881)` is Everest, never `Point(27.9881, 86.925)`. This swap is a classic bug, found even in published tutorials, and shapely cannot warn you about it.

Printing a point shows its text form, `POINT (86.925 27.9881)`, with x first. `.x` and `.y` give the numbers back, `.geom_type` names the kind of shape, and `.coords` holds the coordinate pairs.

When a geometry is the last line of a cell, Jupyter draws it as a small picture.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import shapely
from shapely import LineString, Point, Polygon, box

DATA = Path("..") / "data"
peaks = pd.read_csv(DATA / "nepal_peaks.csv")
trek = pd.read_csv(DATA / "everest_trek.csv")

everest = Point(86.9250, 27.9881)      # Point(longitude, latitude)
print(everest)
print(everest.x, everest.y)
print(everest.geom_type)
print(list(everest.coords))

In [ ]:
row = peaks.iloc[3]                    # the Makalu row
makalu = Point(row["lon"], row["lat"])
makalu

`list(everest.coords)` gives one pair in round brackets, `(86.925, 27.9881)`. Round brackets make a **tuple**: like a list, but it cannot be changed after it is made. Coordinates are often written as tuples.

The picture of Makalu is a single dot: a point has a position but no length and no area.

### Practice

**Q1 · one line:** Make a point `lhotse` for Lhotse, at longitude 86.9333 and latitude 27.9617, and print it.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
lhotse = Point(86.9333, 27.9617)
print(lhotse)
```

It prints `POINT (86.9333 27.9617)`. The first number is always x, the longitude.
</details>

**Q2 · two lines:** Make a point `lukla` from the first row of `trek`, then print its latitude with `.y`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
lukla = Point(trek["lon"].iloc[0], trek["lat"].iloc[0])
print(lukla.y)
```

It prints 27.6866. `.y` is the second number you gave, so the latitude, provided you passed (lon, lat) in that order.
</details>

**Q3 · three lines:** Build a list `peak_points` with one `Point` per peak: loop over `range(len(peaks))`, take each row with `.iloc`, and append a point (Lessons A3 and A10). Print the length of the list and its last point.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
peak_points = []
for i in range(len(peaks)):
    row = peaks.iloc[i]
    peak_points.append(Point(row["lon"], row["lat"]))
print(len(peak_points), peak_points[-1])
```

It prints `8 POINT (83.8203 28.5961)`: the last row is Annapurna I. Lesson B3 shows a one-line way to do this for a whole table.
</details>

## 2. Lines

A **LineString** is a line through a sequence of points. Each point on it is a **vertex** (plural: vertices). Straight segments join the vertices in order.

Make one from a list of (x, y) pairs: `LineString([(86.7299, 27.6866), (86.7130, 27.7400)])`.

To build a line from a table, select the columns in x, y order and turn them into an array: `trek[["lon", "lat"]].to_numpy()`. Each row becomes one (lon, lat) pair, as in a NumPy array from Lesson A8.

The order of the rows is the order of the line. If the table is not in walking order, sort it first.

`len(line.coords)` counts the vertices, and `line.coords[0]` is the first one.

In [ ]:
xy = trek[["lon", "lat"]].to_numpy()
print(xy[:3])

trek_line = LineString(xy)
print(trek_line.geom_type, len(trek_line.coords))
print(trek_line.coords[0], trek_line.coords[-1])
trek_line

The array has one row per stop, with the longitude first. The line has 8 vertices, from Lukla, `(86.7299, 27.6866)`, to Everest Base Camp, `(86.8528, 28.0029)`. The small picture shows the shape of the trail you plotted in Lesson B1.

### Practice

**Q1 · one line:** Print the third vertex of `trek_line`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(trek_line.coords[2])
```

It prints `(86.7139, 27.8047)`: Namche Bazaar. Vertices are counted from 0, like list items.
</details>

**Q2 · two lines:** Make a line `first_days` from the first three stops only (Lukla to Namche Bazaar), using `.iloc[0:3]`, and print it.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
first_days = LineString(trek[["lon", "lat"]].iloc[0:3].to_numpy())
print(first_days)
```

It prints `LINESTRING (86.7299 27.6866, 86.713 27.74, 86.7139 27.8047)`. The slice keeps rows 0, 1 and 2, as in Lesson A9.
</details>

**Q3 · three lines:** Build the line for the walk *down*, from Everest Base Camp to Lukla. Sort `trek` by `order` from largest to smallest, build the line, and print its first vertex.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
down = trek.sort_values("order", ascending=False)
down_line = LineString(down[["lon", "lat"]].to_numpy())
print(down_line.coords[0])
```

The first vertex is now `(86.8528, 28.0029)`, Everest Base Camp. Same stops, opposite direction: a line remembers the order of its vertices.
</details>

## 3. Polygons and boxes

A **Polygon** is an area enclosed by a ring of vertices. List the corners in order, going around the edge. You do not need to repeat the first corner at the end: shapely closes the ring for you.

A rectangle that follows the x and y axes is so common that shapely has a shortcut: `box(min_x, min_y, max_x, max_y)`.

Here is a box around the Kathmandu Valley, from 85.2° to 85.55° E and from 27.55° to 27.82° N. It is approximate, drawn for practice, not an official boundary.

Three handy properties:

- `.exterior.coords`: the corners of the outer ring,
- `.bounds`: `(min_x, min_y, max_x, max_y)`, the smallest box around the shape,
- `.centroid`: the centre of the shape, as a `Point`.

In [ ]:
valley = Polygon([(85.2, 27.55), (85.55, 27.55), (85.55, 27.82), (85.2, 27.82)])
print(valley)

valley_box = box(85.2, 27.55, 85.55, 27.82)
print(valley_box.bounds)
print(valley_box.centroid)
valley_box

The printed polygon has **five** corners: shapely repeated the first one at the end to close the ring. `box()` made the same rectangle from only four numbers. Its centroid is at (85.375, 27.685), halfway between the edges.

### Practice

**Q1 · one line:** How many coordinate pairs are in the outer ring of `valley`?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
len(valley.exterior.coords)
```

5: four corners plus the first corner repeated to close the ring.
</details>

**Q2 · two lines:** Make a triangle `khumbu_triangle` with corners at Everest (86.925, 27.9881), Makalu (87.0889, 27.8897) and Cho Oyu (86.6608, 28.0942). Print its centroid.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
khumbu_triangle = Polygon([(86.925, 27.9881), (87.0889, 27.8897), (86.6608, 28.0942)])
print(khumbu_triangle.centroid)
```

The centroid is about (86.8916, 27.9907), the balance point of the triangle. A polygon can have any number of corners, as long as it has at least three.
</details>

**Q3 · three lines:** Make a box `peaks_box` around all eight peaks, using the smallest and largest `lon` and `lat` in `peaks` (Lesson A9). Print its bounds.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
peaks_box = box(peaks["lon"].min(), peaks["lat"].min(), peaks["lon"].max(), peaks["lat"].max())
print(peaks_box.bounds)
```

It prints `(83.4875, 27.7025, 88.1475, 28.6967)`. The box runs from Dhaulagiri I in the west to Kanchenjunga in the east. Kanchenjunga is also the farthest south, and Dhaulagiri I the farthest north. Bounding boxes like this are used to zoom maps and to search data quickly.
</details>

## 4. Length, area and distance are in degrees

Every geometry can measure itself:

- `.length`: the length of a line, or the perimeter of a polygon,
- `.area`: the area of a polygon,
- `a.distance(b)`: the shortest straight distance between two geometries.

Shapely calculates on a flat plane, in the units of the coordinates. Our coordinates are degrees, so the answers are in **degrees** and **square degrees**.

Those units mean little. In Lesson B1 you saw that a degree of latitude is about 111 km, while a degree of longitude in Nepal is about 98 km. A "degree" is not even a fixed distance. Use these numbers only to compare shapes that lie close together, and never report them. Lesson B4 converts the coordinates to metres first.

**Predict:** what is the area of the valley box, 0.35° wide and 0.27° tall?

In [ ]:
print(trek_line.length)
print(valley_box.area)
print(valley_box.length)
print(everest.distance(makalu))

In [ ]:
# a rough area in km², with the rules from Lesson B1
lat_mid = valley_box.centroid.y
width_km = 0.35 * 111 * math.cos(math.radians(lat_mid))
height_km = 0.27 * 111
print(round(width_km, 1), round(height_km, 1), round(width_km * height_km))

The trek line is 0.397 "degrees" long and the valley box covers 0.0945 square degrees, numbers that no map reader understands. (Python shows tiny binary leftovers such as `0.09449999999999832`.) Everest and Makalu are 0.191 degrees apart.

With the rules from Lesson B1, the box is about 34.4 km wide and 30.0 km tall, so about 1,031 km². That is a rough answer; Lesson B4 gets a precise one.

### Practice

**Q1 · one line:** How far apart are Everest and Lhotse, in degrees? Make Lhotse's point inside the brackets.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
everest.distance(Point(86.9333, 27.9617))
```

About 0.0277 degrees: the two summits are close together. This number cannot be turned into km exactly, because the gap is partly east–west and partly north–south.
</details>

**Q2 · two lines:** Make two boxes of exactly 1° × 1°: one at the equator, `box(0, 0, 1, 1)`, and one in Nepal, `box(86, 28, 87, 29)`. Print the area of each.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(box(0, 0, 1, 1).area)
print(box(86, 28, 87, 29).area)
```

Both areas are 1.0 square degree. On the ground the two boxes are different sizes, as the next question shows, but shapely cannot know that.
</details>

**Q3 · three lines:** Estimate both boxes in km² with the Lesson B1 rules. Use 111 km per degree of latitude, and 111 × cos(latitude) km per degree of longitude. Take each box's middle latitude: 0.5° and 28.5°. Print both, rounded.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
equator_km2 = 111 * 111 * math.cos(math.radians(0.5))
nepal_km2 = 111 * 111 * math.cos(math.radians(28.5))
print(round(equator_km2), round(nepal_km2))
```

About 12,321 km² at the equator but only about 10,828 km² in Nepal: the same "1 square degree" is 12% smaller there. That is why areas in square degrees are meaningless.
</details>

## 5. How shapes relate: contains, within, intersects

Many spatial questions have a yes-or-no answer. Is this peak inside the box? Does the trail cross the district? Shapely answers them with **predicates**: methods that return `True` or `False`.

| Predicate | `True` when |
| --- | --- |
| `a.contains(b)` | b lies completely inside a |
| `a.within(b)` | a lies completely inside b (the reverse of contains) |
| `a.intersects(b)` | a and b touch or overlap at all |

The examples use a box around the Khumbu region, `box(86.6, 27.6, 87.0, 28.1)`, drawn for practice.

**Predict:** is Makalu (87.0889° E) inside the Khumbu box? Is the whole trek line inside it?

In [ ]:
khumbu = box(86.6, 27.6, 87.0, 28.1)

print(khumbu.contains(everest))
print(makalu.within(khumbu))
print(trek_line.within(khumbu))
print(trek_line.intersects(valley_box))

Everest is inside the box. Makalu is not: its longitude, 87.0889°, is east of the box's edge at 87.0°. The whole trek line lies inside the box. The trek line does not touch the Kathmandu Valley box at all.

`a.contains(b)` and `b.within(a)` always give the same answer; choose the one that reads best.

### Practice

**Q1 · one line:** Is Everest inside the Kathmandu Valley box?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
valley_box.contains(everest)
```

`False`. Everest is far to the east of the valley.
</details>

**Q2 · two lines:** Make a point for Namche Bazaar from row 2 of `trek`, then test whether it lies within the Khumbu box.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
namche = Point(trek["lon"].iloc[2], trek["lat"].iloc[2])
print(namche.within(khumbu))
```

`True`: every stop of the trek is inside the box, because the whole line is.
</details>

**Q3 · three lines:** Make a box for the northern part of the trek, `box(86.6, 27.9, 87.0, 28.1)`. Print whether it **contains** the trek line, and whether it **intersects** it. Why are the answers different?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
north_box = box(86.6, 27.9, 87.0, 28.1)
print(north_box.contains(trek_line))
print(north_box.intersects(trek_line))
```

`False`, then `True`. The trail enters the box between Dingboche (27.892° N) and Lobuche (27.948° N), so they intersect. But the trail starts at Lukla, at 27.69° N, outside the box, so the box does not contain the whole line.
</details>

## 6. WKT: geometry as text

Every geometry has a standard text form called **Well-Known Text** (WKT). It is what `print` showed you all lesson:

- `POINT (86.925 27.9881)`
- `LINESTRING (86.7299 27.6866, 86.713 27.74, ...)`
- `POLYGON ((85.55 27.55, 85.55 27.82, ...))`

Inside WKT, a space separates x from y and a comma separates one vertex from the next. A polygon has double brackets, because a polygon can also have holes, and each hole is another ring in its own brackets.

WKT is understood by QGIS, spatial databases and many other programs. It is a handy way to store a shape in one cell of a CSV file, or to send it in a message.

`.wkt` gives the text of a geometry. `shapely.from_wkt(text)` turns text back into a geometry.

In [ ]:
print(everest.wkt)
print(valley_box.wkt)

text = "LINESTRING (86.7299 27.6866, 86.713 27.74, 86.7139 27.8047)"
first_days = shapely.from_wkt(text)
print(first_days.geom_type, len(first_days.coords))

`from_wkt` read the text and made a real `LineString` with 3 vertices, which can measure itself and be compared with other shapes.

### Practice

**Q1 · one line:** Print the WKT of `trek_line`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(trek_line.wkt)
```

It starts `LINESTRING (86.7299 27.6866, 86.713 27.74, ...` and lists all 8 vertices, x first.
</details>

**Q2 · two lines:** Make a geometry from the text `"POINT (84.5597 28.5497)"`, then print its x and y. Which peak is it?

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
mystery = shapely.from_wkt("POINT (84.5597 28.5497)")
print(mystery.x, mystery.y)
```

It prints `84.5597 28.5497`: Manaslu's position in `nepal_peaks.csv`.
</details>

**Q3 · three lines:** A colleague sends the text `"POLYGON ((85.2 27.55, 85.55 27.55, 85.55 27.82, 85.2 27.82, 85.2 27.55))"`. Load it and check whether it is the same shape as `valley_box` with `.equals()`, a predicate that is `True` when two shapes cover exactly the same points. Also print its area.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
shared = shapely.from_wkt("POLYGON ((85.2 27.55, 85.55 27.55, 85.55 27.82, 85.2 27.82, 85.2 27.55))")
print(shared.equals(valley_box))
print(shared.area)
```

`True`, with the same area of about 0.0945 square degrees. The colleague listed the corners starting from a different one, but `.equals` compares the shapes, not the order of the text.
</details>

## 7. Drawing geometries with GeoPandas

Matplotlib draws lists of numbers, not shapely shapes. **GeoPandas**, the main library of Part B, can draw them.

Put geometries in a **GeoSeries**, a pandas Series that holds geometries: `gpd.GeoSeries([valley_box, khumbu], crs="EPSG:4326")`. Then call `.plot()`.

`crs="EPSG:4326"` tells GeoPandas that the numbers are longitude and latitude. It then sets the aspect for you, as you did by hand in Lesson B1. Lesson B4 explains these codes.

`.plot(ax=ax)` draws on your own axes, so you can draw several layers on one map. The layer drawn last is on top. Useful options: `color`, `edgecolor`, `facecolor="none"` (outline only), `linewidth`, `linestyle`, `marker` and `markersize`.

In [ ]:
import geopandas as gpd

fig, ax = plt.subplots(figsize=(6, 6))
gpd.GeoSeries([khumbu], crs="EPSG:4326").plot(ax=ax, facecolor="none", edgecolor="grey", linestyle="--")
gpd.GeoSeries([trek_line], crs="EPSG:4326").plot(ax=ax, color="tab:blue")
gpd.GeoSeries([everest, makalu], crs="EPSG:4326").plot(ax=ax, color="tab:red", marker="^", markersize=60)

ax.set_title("The Khumbu box, the trek and two peaks")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
plt.show()

The grey dashed box is drawn first, so the trek and the peaks sit on top of it. The trek (blue) lies inside the box, and Everest is a little east of the trail's end. Makalu sits outside the box's eastern edge, as `makalu.within(khumbu)` said in step 5.

### Practice

**Q1 · one line:** Draw `valley_box` on its own with GeoPandas.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gpd.GeoSeries([valley_box], crs="EPSG:4326").plot()
```

A filled blue rectangle. `.plot()` without `ax=` makes its own figure, fine for a quick look.
</details>

**Q2 · two lines:** Draw `valley_box` as an outline only, then its centroid on top as a red point.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
gpd.GeoSeries([valley_box], crs="EPSG:4326").plot(ax=ax, facecolor="none", edgecolor="black")
gpd.GeoSeries([valley_box.centroid], crs="EPSG:4326").plot(ax=ax, color="tab:red")
plt.show()
```

Both layers share `ax`, so they appear on one map. The red dot sits in the middle of the rectangle.
</details>

**Q3 · three lines:** Draw `khumbu_triangle` from step 3 (redefine it if needed), the trek line, and Everest on one map, with a title.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
khumbu_triangle = Polygon([(86.925, 27.9881), (87.0889, 27.8897), (86.6608, 28.0942)])
fig, ax = plt.subplots()
gpd.GeoSeries([khumbu_triangle], crs="EPSG:4326").plot(ax=ax, color="lightgrey")
gpd.GeoSeries([trek_line], crs="EPSG:4326").plot(ax=ax, color="tab:blue")
gpd.GeoSeries([everest], crs="EPSG:4326").plot(ax=ax, color="tab:red")
ax.set_title("Triangle between Everest, Makalu and Cho Oyu")
plt.show()
```

The triangle is drawn first, in light grey, so the line and the point stay visible on top. The trail ends a little below the triangle's long edge, west of Everest.
</details>

## 8. When it breaks: a polygon with only two corners

A rectangle is defined by two corners, so it seems natural to give `Polygon` two corners. Run the cell.

In [ ]:
valley_wrong = Polygon([(85.2, 27.55), (85.55, 27.82)])

Read the last line:

`ValueError: A linearring requires at least 4 coordinates.`

- `ValueError`: the values had the right type, but did not make sense together.
- A **linear ring** is the closed boundary of a polygon. It needs at least three different corners, plus the first one repeated to close it: 4 coordinates. Two corners make a line, not an area.

Two fixes: list all four corners, as in step 3, or use `box(85.2, 27.55, 85.55, 27.82)`, which takes the two corners as four numbers.

Remember the more dangerous bug from this lesson: `Point(lat, lon)` raises **no** error at all. Only a map shows it.

**Try it:** fix the cell with `box`.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Which peaks are in the Khumbu box?

Loop over the rows of `peaks`, make a point for each, and print the names of the peaks inside `khumbu`. Count them as you go, and print the count at the end.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
count = 0
for i in range(len(peaks)):
    row = peaks.iloc[i]
    peak = Point(row["lon"], row["lat"])
    if khumbu.contains(peak):
        print(row["name"])
        count += 1
print(count, "peaks are inside the box")
```

Everest, Lhotse and Cho Oyu are inside, so the count is 3. Cho Oyu, at 28.0942° N, is barely south of the box's northern edge at 28.1°; Makalu is a little east of it.
</details>

### Review 2 · Landslides in a study box

Read `practice_landslides.csv`. A made-up study box around Chitwan runs from 84.3° to 84.65° E and from 27.35° to 27.85° N. Count the landslides inside the box, and collect their district names in a list. Print the count and the set of district names with `set(names)`, which keeps each name once.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
slides = pd.read_csv(DATA / "practice_landslides.csv")
study_box = box(84.3, 27.35, 84.65, 27.85)

names = []
for i in range(len(slides)):
    row = slides.iloc[i]
    if study_box.contains(Point(row["lon"], row["lat"])):
        names.append(row["district"])
print(len(names), set(names))
```

9 landslides are inside, all recorded in Chitwan. That matches the 9 Chitwan records you counted in Lesson A9, a good sign that the box and the coordinates agree.
</details>

### Review 3 · The trek, one leg at a time

Build a list `legs` with one two-point `LineString` per leg of the trek (Lukla to Phakding, Phakding to Namche Bazaar, …). Print the number of legs, the sum of their lengths, and `trek_line.length`. Round both lengths to 4 decimal places.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
legs = []
for i in range(len(trek) - 1):
    start = (trek["lon"].iloc[i], trek["lat"].iloc[i])
    end = (trek["lon"].iloc[i + 1], trek["lat"].iloc[i + 1])
    legs.append(LineString([start, end]))

total_deg = 0
for leg in legs:
    total_deg += leg.length
print(len(legs), round(total_deg, 4), round(trek_line.length, 4))
```

There are 7 legs, and their lengths add up to 0.3969 degrees, exactly the length of the whole line: a line's length is the sum of its segments. Eight stops make seven legs, so the loop stops one before the end.
</details>

### Review 4 · One map with every shape

Draw one map with: the Kathmandu Valley box and the Khumbu box as grey outlines, the trek line in blue, and all eight peaks as red triangles. Build the peak points in a loop. Add a title and axis labels with units.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
points = []
for i in range(len(peaks)):
    points.append(Point(peaks["lon"].iloc[i], peaks["lat"].iloc[i]))

fig, ax = plt.subplots(figsize=(10, 4))
gpd.GeoSeries([valley_box, khumbu], crs="EPSG:4326").plot(ax=ax, facecolor="none", edgecolor="grey")
gpd.GeoSeries([trek_line], crs="EPSG:4326").plot(ax=ax, color="tab:blue")
gpd.GeoSeries(points, crs="EPSG:4326").plot(ax=ax, color="tab:red", marker="^", markersize=50)
ax.set_title("Two practice boxes, the Everest trek and the eight peaks")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
plt.show()
```

One GeoSeries can hold several shapes, so both boxes are drawn in one call. The valley box sits a little west of the middle; the trek and the Khumbu peaks are in the east. Without district boundaries the map still lacks context: Lesson B3 adds them.
</details>

## Recap

- **shapely** builds geometries: `Point(lon, lat)`, `LineString([...])`, `Polygon([...])` and `box(min_x, min_y, max_x, max_y)`. Always x = longitude first.
- `.x`, `.y`, `.coords`, `.exterior.coords`, `.bounds` and `.centroid` read a geometry's parts.
- `.length`, `.area` and `.distance()` work in the units of the coordinates: with longitude and latitude that means degrees, which are not fit to report.
- Predicates such as `contains`, `within`, `intersects` and `equals` answer yes-or-no spatial questions.
- **WKT** is the standard text form of a geometry (`.wkt`, `shapely.from_wkt`). A **GeoSeries** with `crs="EPSG:4326"` draws geometries with `.plot(ax=ax)`.

### Words you learned

| Word | Meaning |
| --- | --- |
| shapely | The Python library for points, lines and polygons on a flat plane |
| geometry | A shape object: a point, line or polygon, with its coordinates |
| Point | A geometry with one position and no length or area |
| tuple | A sequence in round brackets, like a list that cannot be changed: `(86.925, 27.9881)` |
| LineString | A line through a sequence of points |
| vertex (vertices) | One of the points that make up a line or polygon |
| Polygon | An area enclosed by a closed ring of vertices |
| linear ring | The closed boundary of a polygon; needs at least 4 coordinates |
| bounds | `(min_x, min_y, max_x, max_y)`: the smallest box around a shape |
| bounding box | A rectangle that follows the axes and just encloses a shape |
| centroid | The centre (balance point) of a shape |
| square degree | The area of a 1° × 1° box; not a fixed area on the ground |
| predicate | A method that answers a spatial question with `True` or `False` |
| contains / within | b is completely inside a / a is completely inside b |
| intersects | Two shapes touch or overlap at all |
| WKT | Well-Known Text: the standard text form of a geometry, such as `POINT (86.925 27.9881)` |
| GeoSeries | A pandas Series that holds geometries and can plot them |

## Next

**B3 · GeoDataFrames and file formats:** load Nepal's 77 district polygons into a table with a geometry column. Then turn CSV files into points, and save your layers as GeoPackage, Shapefile and GeoJSON files.

<!-- pygile-nav -->
**PyGILE Course** · lesson 12 of 21 · [Course home](../README.md)  
← Previous: [B1 · Maps, coordinates, latitude and longitude: placing Nepal on the Earth](B01_Maps_and_coordinates.ipynb) · Next: [B3 · GeoDataFrames and file formats: Nepal's districts as a table](B03_GeoDataFrames_and_file_formats.ipynb) →